# Lab 3: From a table to a map

This notebook turns Montreal Wi-Fi hotspot locations into a file a web map can read, then puts them on a map.

This is the pipeline your project runs on:

```
Data/MTLwifi.csv  ->  this notebook  ->  Data/internet.geojson  ->  Site/  ->  a map in your browser
   (the table)         (Python)         (the web's map format)   (Leaflet)
```

**How this works:** run the cells from top to bottom. The table's location, type, borough, and active-zone fields become feature properties.

## Part 1. Load the active Montreal Wi-Fi locations

This cell reads the CSV, keeps rows marked `X` in `Zone active`, and reports how many inactive or blank rows were removed.

In [47]:
import pandas as pd
import json

df = pd.read_csv("../Data/MTLwifi.csv")
clean = df[df["Zone active"] == "X"].copy()
dropped = len(df) - len(clean)

print("locations read:   ", len(df))
print("inactive removed:", dropped)
print("active kept:     ", len(clean))
clean.head()

locations read:    545
inactive removed: 12
active kept:      533


,ID,Lieu,Type,Arrondissement,Zone active,Latitude,Longitude,X,Y
0,Antenne,311 Place d'Youville,Intersection,Ville-Marie,X,45.501337,-73.555539,300459.716120,5.040159e+06
1,Antenne,Saint Pierre / Des Recollets,Intersection,Ville-Marie,X,45.502034,-73.557959,300270.652886,5.040236e+06
2,Antenne,Saint Paul / Francois Xavier,Intersection,Ville-Marie,X,45.502735,-73.555258,300481.782694,5.040314e+06
3,Antenne,Saint Jean / De L'Hopital,Intersection,Ville-Marie,X,45.502759,-73.557628,300296.577488,5.040317e+06
4,Antenne,Saint Sacrement / Saint Eloi,Intersection,Ville-Marie,X,45.502807,-73.556400,300392.545050,5.040322e+06


## Part 2. One location, by hand

A GeoJSON **feature** is a dictionary with three entries: its `type`, its `geometry` (where it is) and its `properties` (the other information you want to show). Here is one row of the table:

In [48]:
row = clean.iloc[0]
row

ID                             Antenne
Lieu              311 Place d'Youville
Type                      Intersection
Arrondissement             Ville-Marie
Zone active                          X
Latitude                     45.501337
Longitude                   -73.555539
X                         300459.71612
Y                       5040158.739964
Name: 0, dtype: object

Now build the feature for that row by hand. Two things to get right:

1. **Coordinates go longitude first, then latitude.** GeoJSON is `[lon, lat]`. Leaflet's `setView` is `[lat, lon]`. Mixing them up puts the location in the wrong place.
2. **Numbers from pandas need converting** with `float()` before they go into JSON.

Use the row's `Longitude`, `Latitude`, `ID`, `Lieu`, `Type`, `Arrondissement`, and `Zone active` columns.

In [49]:
feature = {
    "type": "Feature",
    "geometry": {
        "type": "Point",
        "coordinates": [float(row["Longitude"]), float(row["Latitude"])],
    },
    "properties": {
        "id": row["ID"],
        "lieu": row["Lieu"],
        "wifi_type": row["Type"],
        "arrondissement": row["Arrondissement"],
        "zone_active": None if pd.isna(row["Zone active"]) else row["Zone active"],
        "latitude": float(row["Latitude"]),
        "longitude": float(row["Longitude"]),
    },
}
print(json.dumps(feature, indent=2))

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      -73.555539,
      45.501337
    ]
  },
  "properties": {
    "id": "Antenne",
    "lieu": "311 Place d'Youville",
    "wifi_type": "Intersection",
    "arrondissement": "Ville-Marie",
    "zone_active": "X",
    "latitude": 45.501337,
    "longitude": -73.555539
  }
}


**Check before you continue.** The first coordinate should be about -73.56 (longitude, west is negative) and the second about 45.50 (latitude). If yours starts with 45, swap them.

## Part 3. Turn it into a function, and test it

You are about to do the same thing for every location. That is what a function is for. Put your dictionary inside `row_to_feature`, so that it takes any row and returns its feature.

Then the test cell checks it on one row you can check by eye. This is the *test* step of the loop: prove a small piece works before you trust it with everything.

In [50]:
def row_to_feature(row):
    """Take one row of the Montreal Wi-Fi table and return one GeoJSON feature."""
    properties = {
        "id": row["ID"],
        "lieu": row["Lieu"],
        "wifi_type": row["Type"],
        "arrondissement": row["Arrondissement"],
        "zone_active": row["Zone active"],
        "latitude": float(row["Latitude"]),
        "longitude": float(row["Longitude"]),
    }
    properties = {
        key: None if pd.isna(value) else value
        for key, value in properties.items()
    }
    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row["Longitude"]), float(row["Latitude"])],
        },
        "properties": properties,
    }

In [51]:
# Test on one row. Every line prints True when the function is right.
test = row_to_feature(clean.iloc[0])
lon, lat = test["geometry"]["coordinates"]
print("is a Feature:          ", test["type"] == "Feature")
print("longitude comes first: ", -74.0 < lon < -73.4)
print("latitude comes second: ", 45.3 < lat < 45.8)
print("has a Wi-Fi type:      ", bool(test["properties"]["wifi_type"]))

is a Feature:           True
longitude comes first:  True
latitude comes second:  True
has a Wi-Fi type:       True


## Part 4. Every location, then write the file

Prediction first. The GeoJSON repeats every property name for every location. Will the output file be **smaller, about the same, or bigger** than the CSV? Roughly how many MB?

**My prediction:** 

Then fill in the loop. `clean.iterrows()` gives you each row in turn; the `_` holds the row number, which you do not need.

In [52]:
features = []
for _, row in clean.iterrows():
    features.append(row_to_feature(row))

collection = {"type": "FeatureCollection", "features": features}

out_path = "../Data/internet.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(collection, f, separators=(",", ":"), allow_nan=False)
    # separators: no spaces, a smaller file.
    # allow_nan=False: stop with an error if any value is NaN (an empty cell).
    # Browsers cannot read NaN in JSON, so without this line the file writes
    # fine and the map fails later with a confusing message. Fail here instead.

import os
print(len(features), "features written to", out_path)
print(round(os.path.getsize(out_path) / 1_000_000, 1), "MB")

533 features written to ../Data/internet.geojson
0.1 MB


## Part 5. Check your output file

This cell reads the GeoJSON back from disk (the way the browser will) and checks it against the Montreal Wi-Fi table and coordinate bounds. Run it.

In [53]:
with open("../Data/internet.geojson", encoding="utf-8") as f:
    back = json.load(f)

problems = []
feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The file is not a FeatureCollection.")
if len(feats) != len(clean):
    problems.append(f"The file has {len(feats)} features but the table has {len(clean)} locations.")
lons = [ft["geometry"]["coordinates"][0] for ft in feats]
lats = [ft["geometry"]["coordinates"][1] for ft in feats]
if feats and not all(-74.0 < x < -73.4 for x in lons):
    problems.append("Some longitudes are outside Montreal. Are longitude and latitude swapped?")
if feats and not all(45.3 < y < 45.8 for y in lats):
    problems.append("Some latitudes are outside Montreal. Are longitude and latitude swapped?")
required_properties = {"id", "lieu", "wifi_type", "arrondissement", "zone_active", "latitude", "longitude"}
if feats and not all(required_properties <= set(ft["properties"]) for ft in feats):
    problems.append("Some features are missing one or more Wi-Fi location properties.")

print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

All checks passed


## Now go to the site

The rest of the lab happens in `site/app.js` and the browser. Follow the handout from Part 6.

---

## Layer B. Your own data through the same pipeline

Copy the cells you need into a new notebook in your **project** folder (see the handout). The only parts that change are the file name, the column names inside `row_to_feature`, and the numbers in the check cell. Everything else is the same pipeline.

## Layer 2. Add tourist attractions

This section reads the tourist-attraction records separately, converts them into GeoJSON, and writes a second file so the existing Wi-Fi data and conversion remain unchanged. The file currently in `Data/` is named `lieux_d_interet (1).csv`.

In [54]:
import pandas as pd
import json

tourist_df = pd.read_csv("../Data/lieux_d_interet (1).csv")
tourist_df = tourist_df[
    tourist_df["Catégorie"].astype(str).str.strip() == "Attrait touristique"
].copy()

tourist_features = []
for _, tourist_row in tourist_df.iterrows():
    latitude = float(tourist_row["Latitude"])
    longitude = float(tourist_row["Longitude"])
    properties = {
        "id": tourist_row["ID"],
        "name": tourist_row["Nom français"],
        "type": tourist_row["Type"],
        "category": tourist_row["Catégorie"],
        "borough": tourist_row["Arrondissement"],
        "latitude": latitude,
        "longitude": longitude,
    }
    properties = {
        key: None if pd.isna(value) else value
        for key, value in properties.items()
    }
    tourist_features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [longitude, latitude],
        },
        "properties": properties,
    })

tourist_collection = {
    "type": "FeatureCollection",
    "features": tourist_features,
}
tourist_out_path = "../Data/tourist_attractions.geojson"
with open(tourist_out_path, "w", encoding="utf-8") as f:
    json.dump(tourist_collection, f, separators=(",", ":"), allow_nan=False)

print(len(tourist_features), "tourist attractions written to", tourist_out_path)

18 tourist attractions written to ../Data/tourist_attractions.geojson


In [55]:
import geopandas as gpd
import numpy as np
from shapely.geometry import box

# Use the active Wi-Fi rows already loaded from MTLwifi.csv.
wifi_points = gpd.GeoDataFrame(
    clean.copy(),
    geometry=gpd.points_from_xy(clean["Longitude"], clean["Latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:32188")  # Montreal projected CRS; units are metres.

GRID_SIZE_M = 1000

def make_grid(gdf, size_m):
    """Create square cells of the requested size covering the Wi-Fi point extent."""
    minx, miny, maxx, maxy = gdf.total_bounds
    min_col, max_col = int(np.floor(minx / size_m)), int(np.floor(maxx / size_m))
    min_row, max_row = int(np.floor(miny / size_m)), int(np.floor(maxy / size_m))
    cells = []
    for row_index in range(min_row, max_row + 1):
        for col_index in range(min_col, max_col + 1):
            cells.append({
                "cell_id": len(cells),
                "grid_row": row_index,
                "grid_col": col_index,
                "geometry": box(
                    col_index * size_m,
                    row_index * size_m,
                    (col_index + 1) * size_m,
                    (row_index + 1) * size_m,
                ),
            })
    return gpd.GeoDataFrame(cells, geometry="geometry", crs=gdf.crs)

grid = make_grid(wifi_points, GRID_SIZE_M)
wifi_points["grid_col"] = np.floor(wifi_points.geometry.x / GRID_SIZE_M).astype(int)
wifi_points["grid_row"] = np.floor(wifi_points.geometry.y / GRID_SIZE_M).astype(int)
counts = (
    wifi_points.groupby(["grid_row", "grid_col"])
    .size()
    .rename("wifi_count")
    .reset_index()
)
grid = grid.merge(counts, on=["grid_row", "grid_col"], how="left")
grid["wifi_count"] = grid["wifi_count"].fillna(0).astype(int)

def wifi_category(count):
    if count == 0:
        return "0 spots"
    if count <= 20:
        return "1–20 spots"
    if count <= 30:
        return "21–30 spots"
    if count <= 40:
        return "31–40 spots"
    return "41+ spots"

grid["category"] = grid["wifi_count"].map(wifi_category)
grid_out_path = "../Data/wifi_grid.geojson"
grid.to_crs("EPSG:4326")[["cell_id", "wifi_count", "category", "geometry"]].to_file(
    grid_out_path, driver="GeoJSON", index=False
)
print(len(grid), f"{GRID_SIZE_M}-metre cells written to", grid_out_path)
print(grid["category"].value_counts().sort_index())

1221 1000-metre cells written to ../Data/wifi_grid.geojson
category
0 spots        1046
1–20 spots      171
21–30 spots       3
31–40 spots       1
Name: count, dtype: int64
